# Section A - Setup

In [12]:
# Cell 1 — Install dependencies for Stage 0 (leaf gate)
!pip install -q ultralytics

import os
import json
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO, YOLOE

print("Torch CUDA available:", torch.cuda.is_available())

Torch CUDA available: True


In [2]:
# Cell 1b — Reusable safe-JSON helper (numpy types aren't natively JSON serializable)
def json_safe_default(o):
    if isinstance(o, np.generic):
        return o.item()
    raise TypeError(f'Object of type {o.__class__.__name__} is not JSON serializable')

In [3]:
# Cell 2 — Mount Drive and set Stage 0 paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/CropClassifier_v2')
STAGE0_ROOT = DRIVE_ROOT / 'stage0_leaf_gate'
STAGE0_ROOT.mkdir(parents=True, exist_ok=True)

MODELS_DIR = STAGE0_ROOT / 'models'
CONFIG_DIR = STAGE0_ROOT / 'config'
RESULTS_DIR = STAGE0_ROOT / 'results'
for d in (MODELS_DIR, CONFIG_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Stage 0 root:", STAGE0_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Stage 0 root: /content/drive/MyDrive/CropClassifier_v2/stage0_leaf_gate


In [4]:
# Cell 3 — Load pretrained open-vocabulary detector (Drive-cached, resume-safe)
from ultralytics import YOLOE

MODEL_FILENAME = 'yoloe-11s-seg.pt'  # small/fast variant, good fit for a gating stage
DRIVE_MODEL_PATH = MODELS_DIR / MODEL_FILENAME

if DRIVE_MODEL_PATH.exists():
    print(f"Found cached weights on Drive: {DRIVE_MODEL_PATH}")
    model = YOLOE(str(DRIVE_MODEL_PATH))
else:
    print("No cached weights found — downloading fresh copy...")
    model = YOLOE(MODEL_FILENAME)

    import shutil
    local_cache_path = Path(MODEL_FILENAME)
    if local_cache_path.exists():
        shutil.copy(local_cache_path, DRIVE_MODEL_PATH)
        print(f"Cached weights to Drive: {DRIVE_MODEL_PATH}")
    else:
        print("Warning: could not locate downloaded weights file to cache — check cwd/cache dir after this runs.")

print(type(model))

Found cached weights on Drive: /content/drive/MyDrive/CropClassifier_v2/stage0_leaf_gate/models/yoloe-11s-seg.pt
<class 'ultralytics.models.yolo.model.YOLOE'>


# Section B - Vocab Setup

In [5]:
# Cell 4 (fix) — force regeneration since VOCAB has changed since the cache was written

VOCAB_CONFIG_PATH = CONFIG_DIR / 'vocab_config.json'
EMBEDDINGS_PATH = MODELS_DIR / 'leaf_vocab_embeddings.pt'

VOCAB = [
    "leaf", "plant leaf", "tree leaf",
    "grass leaf", "blade leaf", "crop leaf", "leaf in hand",
    "human hand", "person", "face",
    "sky", "soil", "ground",
    "mobile phone", "shoe", "clothing",
    "vehicle", "animal", "tool",
    "building", "wall", "sack of grain",
]

with open(VOCAB_CONFIG_PATH, 'r') as f:
    cached_vocab = json.load(f).get('vocab', [])

if cached_vocab != VOCAB:
    print("VOCAB has changed since cache was written — regenerating embeddings...")
    text_pe = model.get_text_pe(VOCAB)
    model.set_classes(VOCAB, text_pe)

    with open(VOCAB_CONFIG_PATH, 'w') as f:
        json.dump({'vocab': VOCAB}, f, indent=2)
    torch.save(text_pe, EMBEDDINGS_PATH)
    print("Regenerated and cached new vocab.")
else:
    text_pe = torch.load(EMBEDDINGS_PATH)
    model.set_classes(VOCAB, text_pe)
    print("Cache matches current VOCAB — loaded as-is.")

print("Current model vocabulary:", model.names)

Cache matches current VOCAB — loaded as-is.
Current model vocabulary: {0: 'leaf', 1: 'plant leaf', 2: 'tree leaf', 3: 'grass leaf', 4: 'blade leaf', 5: 'crop leaf', 6: 'leaf in hand', 7: 'human hand', 8: 'person', 9: 'face', 10: 'sky', 11: 'soil', 12: 'ground', 13: 'mobile phone', 14: 'shoe', 15: 'clothing', 16: 'vehicle', 17: 'animal', 18: 'tool', 19: 'building', 20: 'wall', 21: 'sack of grain'}


# Section C - Validation on existing images

In [6]:
# Cell 5 — Gather image paths for Stage 0 validation
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Reuse your existing OOD folder from Stage 1 (leaf/crop images — positive cases)
OOD_ROOT = Path('/content/drive/MyDrive/disease_detection/unseen/Diseased_and_healthy')

all_ood_images = [
    p for p in OOD_ROOT.rglob('*')
    if p.suffix.lower() in IMG_EXTS and p.is_file()
]

print(f"Found {len(all_ood_images)} OOD (leaf/crop) images for validation")
print("Sample:", all_ood_images[:3])

Found 331 OOD (leaf/crop) images for validation
Sample: [PosixPath('/content/drive/MyDrive/disease_detection/unseen/Diseased_and_healthy/apple/Apple_1.png'), PosixPath('/content/drive/MyDrive/disease_detection/unseen/Diseased_and_healthy/apple/Apple_2.png'), PosixPath('/content/drive/MyDrive/disease_detection/unseen/Diseased_and_healthy/apple/Apple_3.png')]


In [7]:
# Cell 6 (fix) — reset validation results since vocab changed, re-run full inference
import hashlib

vocab_hash = hashlib.md5(str(VOCAB).encode()).hexdigest()[:8]
RESULTS_PATH = RESULTS_DIR / f'stage0_validation_results_{vocab_hash}.json'

# Cell 6 (patched) — guard against empty/corrupted results file
if RESULTS_PATH.exists():
    try:
        with open(RESULTS_PATH, 'r') as f:
            content = f.read().strip()
        if content:
            validation_results = json.loads(content)
            print(f"Resuming — {len(validation_results)} images already processed under this vocab")
        else:
            print("Results file exists but is empty — starting fresh")
            validation_results = {}
    except json.JSONDecodeError:
        print("Results file exists but is corrupted — starting fresh")
        validation_results = {}
else:
    validation_results = {}
    print("No results yet for this vocab — starting fresh")

CHECKPOINT_EVERY = 50

for i, img_path in enumerate(all_ood_images):
    key = str(img_path)
    if key in validation_results:
        continue

    try:
        results = model.predict(str(img_path), conf=0.01, verbose=False)  # low conf, filter later
        r = results[0]

        detections = []
        for box in r.boxes:
            cls_id = int(box.cls.cpu().numpy()[0])
            conf = float(box.conf.cpu().numpy()[0])
            xyxy = box.xyxy.cpu().numpy()[0].tolist()
            detections.append({
                'class_name': model.names[cls_id],
                'confidence': round(conf, 4),
                'box': xyxy,
            })
        detections.sort(key=lambda d: d['confidence'], reverse=True)
        top_detection = detections[0] if detections else None

        validation_results[key] = {
            'true_label': img_path.parent.name,
            'detections': detections,
            'top_call': top_detection['class_name'] if top_detection else 'no_detection',
            'top_confidence': top_detection['confidence'] if top_detection else 0.0,
        }
    except Exception as e:
        validation_results[key] = {'error': str(e)}

    if (i + 1) % CHECKPOINT_EVERY == 0:
        with open(RESULTS_PATH, 'w') as f:
            json.dump(validation_results, f, indent=2, default=json_safe_default)
        print(f"Checkpointed at {i + 1}/{len(all_ood_images)}")

with open(RESULTS_PATH, 'w') as f:
    json.dump(validation_results, f, indent=2, default=json_safe_default)

print(f"Done. Total processed: {len(validation_results)}")

Resuming — 331 images already processed under this vocab
Done. Total processed: 331


In [8]:
# Cell 6b — Inspect full detection lists for a sample of "tree"-confused images
LEAF_TERMS_CHECK = {"leaf", "plant leaf", "tree leaf"}

tree_confused = [
    (k, v) for k, v in validation_results.items()
    if v.get('top_call') == 'tree'
]

print(f"Inspecting {min(5, len(tree_confused))} of {len(tree_confused)} 'tree'-confused images:\n")

for key, v in tree_confused[:5]:
    print(f"Image: {Path(key).name}")
    for d in v['detections'][:5]:  # top 5 detections per image
        flag = " <-- leaf term" if d['class_name'] in LEAF_TERMS_CHECK else ""
        print(f"  {d['class_name']:<15} conf={d['confidence']:.4f}{flag}")
    print()

Inspecting 0 of 0 'tree'-confused images:



In [9]:
# Cell 6c — Re-check "tree"-confused images under the new leaf-priority rule
LEAF_TERMS_CHECK = {"leaf", "plant leaf", "tree leaf"}
CONF_THRESH = 0.08

recovered = 0
still_wrong = 0
for key, v in tree_confused:
    call = resolve_call(v['detections'], LEAF_TERMS_CHECK, CONF_THRESH)
    if call and call['class_name'] in LEAF_TERMS_CHECK:
        recovered += 1
    else:
        still_wrong += 1

print(f"Recovered as leaf under new rule: {recovered}/{len(tree_confused)}")
print(f"Still not resolved as leaf: {still_wrong}/{len(tree_confused)}")

Recovered as leaf under new rule: 0/0
Still not resolved as leaf: 0/0


In [10]:
# Cell 6c — Leaf-priority decision rule
def resolve_call(detections, leaf_terms, threshold):
    """Leaf wins if ANY leaf-term detection clears the threshold, regardless of rank."""
    above_threshold = [d for d in detections if d['confidence'] >= threshold]
    if not above_threshold:
        return None  # no_object_detected

    leaf_matches = [d for d in above_threshold if d['class_name'] in leaf_terms]
    if leaf_matches:
        return max(leaf_matches, key=lambda d: d['confidence'])

    return max(above_threshold, key=lambda d: d['confidence'])

In [11]:
# Cell 6d — Check if remaining confusions are leaf-priority-recoverable
LEAF_TERMS_CHECK = {"leaf", "plant leaf", "tree leaf"}
CONF_THRESH = 0.08

confused = [
    (k, v) for k, v in validation_results.items()
    if v.get('top_call') not in LEAF_TERMS_CHECK and v.get('top_call') != 'no_detection'
]

recovered = 0
still_wrong = 0
recovery_breakdown = Counter()

for key, v in confused:
    call = resolve_call(v['detections'], LEAF_TERMS_CHECK, CONF_THRESH)
    if call and call['class_name'] in LEAF_TERMS_CHECK:
        recovered += 1
    else:
        still_wrong += 1
        recovery_breakdown[v['top_call']] += 1

print(f"Total confused images: {len(confused)}")
print(f"Recovered as leaf under leaf-priority rule: {recovered}")
print(f"Still not resolved as leaf: {still_wrong}")
print("\nBreakdown of images still wrong after fix:")
for cls_name, count in recovery_breakdown.most_common():
    print(f"  {cls_name}: {count}")

NameError: name 'Counter' is not defined

In [13]:
# Cell 6e — Check leaf-term confidence in the 41 still-unresolved images
LEAF_TERMS_CHECK = {"leaf", "plant leaf", "tree leaf"}

still_wrong_keys = [
    k for k, v in confused
    if resolve_call(v['detections'], LEAF_TERMS_CHECK, CONF_THRESH) is None
    or resolve_call(v['detections'], LEAF_TERMS_CHECK, CONF_THRESH)['class_name'] not in LEAF_TERMS_CHECK
]

print(f"Checking {len(still_wrong_keys)} unresolved images for any leaf-term signal:\n")

for key in still_wrong_keys[:15]:  # sample
    v = validation_results[key]
    leaf_dets = [d for d in v['detections'] if d['class_name'] in LEAF_TERMS_CHECK]
    top_leaf_conf = max([d['confidence'] for d in leaf_dets], default=0.0)
    print(f"{Path(key).name:30s} top_call={v['top_call']:15s} best_leaf_conf={top_leaf_conf:.4f}")

Checking 99 unresolved images for any leaf-term signal:

Apple_1.png                    top_call=grass leaf      best_leaf_conf=0.0000
Apple_3.png                    top_call=grass leaf      best_leaf_conf=0.0000
Apple_5.png                    top_call=grass leaf      best_leaf_conf=0.0000
Apple_6.png                    top_call=ground          best_leaf_conf=0.0538
Apple_7.png                    top_call=crop leaf       best_leaf_conf=0.0597
IMG_4139.JPG                   top_call=clothing        best_leaf_conf=0.0765
IMG_4145.JPG                   top_call=sky             best_leaf_conf=0.0708
IMG_4137.JPG                   top_call=person          best_leaf_conf=0.0000
Screenshot 2026-06-29 at 11.08.17 AM.png top_call=crop leaf       best_leaf_conf=0.0000
Screenshot 2026-06-29 at 11.08.45 AM.png top_call=blade leaf      best_leaf_conf=0.0000
Screenshot 2026-06-29 at 11.11.01 AM.png top_call=blade leaf      best_leaf_conf=0.0237
Screenshot 2026-06-29 at 11.11.42 AM.png top_call=perso

In [ ]:
# Cell 6f — Visualize unresolved images, split by whether leaf signal exists
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Recompute full list with leaf confidence for all 41 (not just the printed sample of 15)
unresolved_info = []
for key in still_wrong_keys:
    v = validation_results[key]
    leaf_dets = [d for d in v['detections'] if d['class_name'] in LEAF_TERMS_CHECK]
    top_leaf_conf = max([d['confidence'] for d in leaf_dets], default=0.0)
    unresolved_info.append({
        'key': key,
        'top_call': v['top_call'],
        'leaf_conf': top_leaf_conf,
    })

zero_signal = [u for u in unresolved_info if u['leaf_conf'] == 0.0]
low_signal = [u for u in unresolved_info if u['leaf_conf'] > 0.0]

print(f"True zero leaf signal: {len(zero_signal)}")
print(f"Some leaf signal (just below threshold): {len(low_signal)}")

def plot_grid(items, title, cols=5):
    n = len(items)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.3))
    axes = axes.flatten()

    for i, item in enumerate(items):
        img = mpimg.imread(item['key'])
        axes[i].imshow(img)
        label = f"{Path(item['key']).name}\ncall={item['top_call']}\nleaf_conf={item['leaf_conf']:.3f}"
        axes[i].set_title(label, fontsize=7)
        axes[i].axis('off')

    for j in range(n, len(axes)):
        axes[j].axis('off')

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{title.replace(' ', '_').lower()}.png", dpi=100)
    plt.show()

plot_grid(zero_signal, "Zero Leaf Signal")
plot_grid(low_signal, "Low Leaf Signal (Below Threshold)")

In [15]:
# |# Cell 7 — Summarize results: leaf-call accuracy on known-leaf images
LEAF_TERMS = {"leaf", "plant leaf", "tree leaf"}

total = len(validation_results)
correct_leaf_calls = sum(
    1 for v in validation_results.values()
    if v.get('top_call') in LEAF_TERMS
)
no_detection = sum(
    1 for v in validation_results.values()
    if v.get('top_call') == 'no_detection'
)
misclassified = total - correct_leaf_calls - no_detection

print(f"Total images: {total}")
print(f"Correctly called leaf: {correct_leaf_calls} ({correct_leaf_calls/total*100:.1f}%)")
print(f"No detection at all: {no_detection} ({no_detection/total*100:.1f}%)")
print(f"Misclassified as something else: {misclassified} ({misclassified/total*100:.1f}%)")

# Show what it's confusing leaf images with, if anything
from collections import Counter
wrong_calls = Counter(
    v['top_call'] for v in validation_results.values()
    if v.get('top_call') not in LEAF_TERMS and v.get('top_call') != 'no_detection'
)
print("\nTop confusions (leaf images called something else):")
for cls_name, count in wrong_calls.most_common(10):
    print(f"  {cls_name}: {count}")

Total images: 331
Correctly called leaf: 172 (52.0%)
No detection at all: 0 (0.0%)
Misclassified as something else: 159 (48.0%)

Top confusions (leaf images called something else):
  blade leaf: 54
  person: 29
  crop leaf: 15
  grass leaf: 12
  leaf in hand: 12
  sky: 9
  ground: 8
  human hand: 5
  soil: 4
  wall: 4


In [ ]:
# Cell 8 — Visualize images where Stage 0 detected nothing
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

no_detection_keys = [
    k for k, v in validation_results.items()
    if v.get('top_call') == 'no_detection'
]

print(f"Total no-detection images: {len(no_detection_keys)}")

# Show a grid of the first N — adjust N/grid size as needed
N = 20
cols = 5
rows = (N + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten()

for i, key in enumerate(no_detection_keys[:N]):
    img = mpimg.imread(key)
    axes[i].imshow(img)
    axes[i].set_title(Path(key).parent.name, fontsize=8)
    axes[i].axis('off')

# Hide any unused subplot axes
for j in range(len(no_detection_keys[:N]), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'no_detection_sample_grid.png', dpi=100)
plt.show()

In [17]:
# Cell 9 — Check if "no detection" images are borderline or true misses
LOW_CONF = 0.01  # very permissive, just to see if *anything* fires

borderline_results = []

for key in no_detection_keys[:N]:  # same sample as Cell 8, or use full list
    results = model.predict(key, conf=LOW_CONF, verbose=False)
    r = results[0]

    if len(r.boxes) > 0:
        top_box = max(r.boxes, key=lambda b: float(b.conf.cpu().numpy()[0]))
        cls_id = int(top_box.cls.cpu().numpy()[0])
        conf = float(top_box.conf.cpu().numpy()[0])
        borderline_results.append({
            'image': key,
            'class_name': model.names[cls_id],
            'confidence': round(conf, 4),
        })
    else:
        borderline_results.append({'image': key, 'class_name': 'still_none', 'confidence': 0.0})

for r in borderline_results:
    print(f"{Path(r['image']).name:40s} -> {r['class_name']:15s} conf={r['confidence']}")

In [18]:
# Cell 10 (fixed) — cast numpy types to native Python types before JSON serialization
def analyze_background(img_path, border_pct=0.05):
    img = Image.open(img_path)
    mode = img.mode
    arr = np.array(img.convert('RGB'))
    h, w = arr.shape[:2]
    bh, bw = int(h * border_pct), int(w * border_pct)

    border_pixels = np.concatenate([
        arr[:bh].reshape(-1, 3),
        arr[-bh:].reshape(-1, 3),
        arr[:, :bw].reshape(-1, 3),
        arr[:, -bw:].reshape(-1, 3),
    ])

    std_dev = float(border_pixels.std())
    mean_val = float(border_pixels.mean())
    is_plain_bg = bool(std_dev < 15)  # cast explicitly — numpy bool breaks json.dump

    return {
        'mode': mode,
        'border_std': round(std_dev, 2),
        'border_mean': round(mean_val, 2),
        'likely_plain_bg': is_plain_bg,
    }

In [19]:
# Cell 11 — Check if no-detection images are dominated by full-frame close-ups
# Heuristic: compare border region variance to full-image variance.
# If the leaf's texture/color extends uniformly all the way to the edges
# (no distinguishable background band), it's likely an edge-to-edge close-up.

# Safe JSON dump helper — handles numpy bool/int/float automatically
def json_safe_default(o):
    if isinstance(o, np.generic):
        return o.item()
    raise TypeError(f'Object of type {o.__class__.__name__} is not JSON serializable')

def save_results():
    with open(RESULTS_PATH, 'w') as f:
        json.dump(validation_results, f, indent=2, default=json_safe_default)


def analyze_frame_fill(img_path, border_pct=0.05):
    arr = np.array(Image.open(img_path).convert('RGB'))
    h, w = arr.shape[:2]
    bh, bw = int(h * border_pct), int(w * border_pct)

    border_pixels = np.concatenate([
        arr[:bh].reshape(-1, 3),
        arr[-bh:].reshape(-1, 3),
        arr[:, :bw].reshape(-1, 3),
        arr[:, -bw:].reshape(-1, 3),
    ])
    center = arr[bh:-bh, bw:-bw].reshape(-1, 3)

    # If border and center have very similar color distributions,
    # there's likely no distinct "background" — subject fills the frame.
    border_mean = border_pixels.mean(axis=0)
    center_mean = center.mean(axis=0)
    color_diff = float(np.linalg.norm(border_mean - center_mean))

    return {'color_diff_border_vs_center': round(color_diff, 2)}

fill_diffs = []
for key in no_detection_keys:
    info = analyze_frame_fill(key)
    validation_results[key]['frame_fill_analysis'] = info
    fill_diffs.append(info['color_diff_border_vs_center'])

fill_diffs = np.array(fill_diffs)
print(f"Median border-vs-center color diff (no-detection set): {np.median(fill_diffs):.2f}")
print(f"Images with very low diff (<10, likely edge-to-edge fill): {int((fill_diffs < 10).sum())}")

for k in validation_results:
    if 'frame_fill_analysis' in validation_results[k]:
        validation_results[k]['frame_fill_analysis']['color_diff_border_vs_center'] = float(
            validation_results[k]['frame_fill_analysis']['color_diff_border_vs_center']
        )

with open(RESULTS_PATH, 'w') as f:
   save_results()

Median border-vs-center color diff (no-detection set): nan
Images with very low diff (<10, likely edge-to-edge fill): 0


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [20]:
# Cell 12 — Summarize Cell 9's borderline results
from collections import Counter

conf_buckets = Counter()
for r in borderline_results:
    c = r['confidence']
    if c == 0.0:
        conf_buckets['true_zero (still_none)'] += 1
    elif c < 0.05:
        conf_buckets['0.00–0.05'] += 1
    elif c < 0.10:
        conf_buckets['0.05–0.10'] += 1
    elif c < 0.15:
        conf_buckets['0.10–0.15'] += 1
    else:
        conf_buckets['0.15+ (should have been caught)'] += 1

for bucket, count in conf_buckets.items():
    print(f"{bucket}: {count}")

In [21]:
# Cell 13 — Run low-confidence check across ALL 100 no-detection images
LOW_CONF = 0.01
full_borderline_results = []

for key in no_detection_keys:  # all 100, not just sample
    results = model.predict(key, conf=LOW_CONF, verbose=False)
    r = results[0]

    if len(r.boxes) > 0:
        top_box = max(r.boxes, key=lambda b: float(b.conf.cpu().numpy()[0]))
        cls_id = int(top_box.cls.cpu().numpy()[0])
        conf = float(top_box.conf.cpu().numpy()[0])
        full_borderline_results.append({
            'image': key, 'class_name': model.names[cls_id], 'confidence': round(conf, 4)
        })
    else:
        full_borderline_results.append({'image': key, 'class_name': 'still_none', 'confidence': 0.0})

# Save so this survives a disconnect
with open(RESULTS_DIR / 'full_borderline_results.json', 'w') as f:
    json.dump(full_borderline_results, f, indent=2, default=json_safe_default)

# Distribution
conf_buckets = Counter()
for r in full_borderline_results:
    c = r['confidence']
    if c == 0.0:
        conf_buckets['true_zero (still_none)'] += 1
    elif c < 0.05:
        conf_buckets['0.00–0.05'] += 1
    elif c < 0.10:
        conf_buckets['0.05–0.10'] += 1
    elif c < 0.15:
        conf_buckets['0.10–0.15'] += 1
    else:
        conf_buckets['0.15+ (should have been caught originally)'] += 1

for bucket, count in conf_buckets.items():
    print(f"{bucket}: {count}")

In [22]:
# Cell 14 — Download a small labeled negative set from Open Images V7 via FiftyOne
!pip install -q fiftyone

import fiftyone as fo
import fiftyone.zoo as foz

NEGATIVE_DIR = STAGE0_ROOT / 'negatives'
NEGATIVE_DIR.mkdir(parents=True, exist_ok=True)

NEG_MANIFEST_PATH = RESULTS_DIR / 'negative_manifest.json'

# Classes chosen to roughly match your expanded VOCAB's non-leaf categories
NEG_CLASSES = ["Human hand", "Human face", "Mobile phone", "Footwear", "Tool", "Person"]
SAMPLES_PER_CLASS = 15  # ~90 total images, adjust as needed

if NEG_MANIFEST_PATH.exists():
    print("Found existing negative manifest — skipping re-download")
    with open(NEG_MANIFEST_PATH, 'r') as f:
        neg_manifest = json.load(f)
else:
    neg_manifest = []

    for cls in NEG_CLASSES:
        cls_dir = NEGATIVE_DIR / cls.replace(" ", "_")
        cls_dir.mkdir(parents=True, exist_ok=True)

        # Skip if already populated (resume-safe per class too)
        existing = list(cls_dir.glob('*'))
        if len(existing) >= SAMPLES_PER_CLASS:
            print(f"Already have {len(existing)} images for '{cls}', skipping")
            for p in existing[:SAMPLES_PER_CLASS]:
                neg_manifest.append({'path': str(p), 'true_label': cls})
            continue

        print(f"Downloading samples for class: {cls}")
        dataset = foz.load_zoo_dataset(
            "open-images-v7",
            split="validation",
            label_types=["detections"],
            classes=[cls],
            max_samples=SAMPLES_PER_CLASS,
            dataset_name=f"neg_{cls.replace(' ', '_')}",
        )

        for sample in dataset:
            src_path = Path(sample.filepath)
            dst_path = cls_dir / src_path.name
            if not dst_path.exists():
                import shutil
                shutil.copy(src_path, dst_path)
            neg_manifest.append({'path': str(dst_path), 'true_label': cls})

    with open(NEG_MANIFEST_PATH, 'w') as f:
        json.dump(neg_manifest, f, indent=2, default=json_safe_default)

print(f"Total negative images ready: {len(neg_manifest)}")

Found existing negative manifest — skipping re-download
Total negative images ready: 90


In [23]:
# Cell 15 — Run Stage 0 inference on negative (non-leaf) images
NEG_RESULTS_PATH = RESULTS_DIR / 'negative_validation_results.json'

if NEG_RESULTS_PATH.exists():
    with open(NEG_RESULTS_PATH, 'r') as f:
        neg_results = json.load(f)
    print(f"Resuming — {len(neg_results)} negative images already processed")
else:
    neg_results = {}

LOW_CONF = 0.01  # capture everything, filter by threshold later in analysis

CHECKPOINT_EVERY = 30

for i, entry in enumerate(neg_manifest):
    key = entry['path']
    if key in neg_results:
        continue

    try:
        results = model.predict(key, conf=LOW_CONF, verbose=False)
        r = results[0]

        detections = []
        for box in r.boxes:
            cls_id = int(box.cls.cpu().numpy()[0])
            conf = float(box.conf.cpu().numpy()[0])
            detections.append({
                'class_name': model.names[cls_id],
                'confidence': round(conf, 4),
            })
        detections.sort(key=lambda d: d['confidence'], reverse=True)
        top_detection = detections[0] if detections else None

        neg_results[key] = {
            'true_label': entry['true_label'],
            'detections': detections,
            'top_call': top_detection['class_name'] if top_detection else 'no_detection',
            'top_confidence': top_detection['confidence'] if top_detection else 0.0,
        }
    except Exception as e:
        neg_results[key] = {'error': str(e)}

    if (i + 1) % CHECKPOINT_EVERY == 0:
        with open(NEG_RESULTS_PATH, 'w') as f:
            json.dump(neg_results, f, indent=2, default=json_safe_default)
        print(f"Checkpointed at {i + 1}/{len(neg_manifest)}")

with open(NEG_RESULTS_PATH, 'w') as f:
    json.dump(neg_results, f, indent=2, default=json_safe_default)

print(f"Done. Total negative images processed: {len(neg_results)}")

Resuming — 90 negative images already processed
Done. Total negative images processed: 90


# Section D - Threshold Tuning

In [25]:
# Cell 16 (final) — Precision/recall table using resolve_call() + expanded LEAF_TERMS
LEAF_TERMS = {
    "leaf", "plant leaf", "tree leaf",
    "grass leaf", "blade leaf", "crop leaf", "leaf in hand",
}
CANDIDATE_THRESHOLDS = [0.03, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20]

print(f"{'Threshold':<10} {'Leaf Recall':<15} {'False Positive Rate (neg set)':<30}")
print("-" * 60)

for thresh in CANDIDATE_THRESHOLDS:
    # Leaf recall on positive (OOD) set
    leaf_correct = 0
    leaf_total = 0
    for v in validation_results.values():
        if 'detections' not in v:
            continue
        leaf_total += 1
        call = resolve_call(v['detections'], LEAF_TERMS, thresh)
        if call and call['class_name'] in LEAF_TERMS:
            leaf_correct += 1

    # False positive rate on negative set
    neg_false_positive = 0
    neg_total = 0
    for v in neg_results.values():
        if 'detections' not in v:
            continue
        neg_total += 1
        call = resolve_call(v['detections'], LEAF_TERMS, thresh)
        if call and call['class_name'] in LEAF_TERMS:
            neg_false_positive += 1

    leaf_recall = leaf_correct / leaf_total * 100 if leaf_total else 0
    fp_rate = neg_false_positive / neg_total * 100 if neg_total else 0

    print(f"{thresh:<10} {leaf_recall:<15.1f} {fp_rate:<30.1f}")

Threshold  Leaf Recall     False Positive Rate (neg set) 
------------------------------------------------------------
0.03       97.0            6.7                           
0.05       94.9            3.3                           
0.08       89.1            1.1                           
0.1        85.5            0.0                           
0.12       82.8            0.0                           
0.15       77.6            0.0                           
0.2        68.0            0.0                           


In [ ]:
# # Cell 16 (fixed) — Precision/recall table using un-truncated confidence data
# LEAF_TERMS = {"leaf", "plant leaf", "tree leaf"}
# CANDIDATE_THRESHOLDS = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]

# # Build a single, low-conf-complete list of (confidence, top_class) for ALL 331 positives.
# # For the 231 already correctly called at conf=0.15, reuse their top_confidence.
# # For the 100 no-detection images, use full_borderline_results (run at conf=0.01).

# full_borderline_lookup = {r['image']: r for r in full_borderline_results}

# positive_records = []
# for key, v in validation_results.items():
#     if v.get('top_call') != 'no_detection':
#         positive_records.append({
#             'class_name': v['top_call'],
#             'confidence': v['top_confidence'],
#         })
#     else:
#         # pull the low-conf rerun result instead of the truncated one
#         low_conf_entry = full_borderline_lookup.get(key)
#         if low_conf_entry:
#             positive_records.append({
#                 'class_name': low_conf_entry['class_name'],
#                 'confidence': low_conf_entry['confidence'],
#             })

# print(f"Total positive records reconstructed: {len(positive_records)}")

# print(f"\n{'Threshold':<10} {'Leaf Recall':<15} {'False Positive Rate (neg set)':<30}")
# print("-" * 60)

# for thresh in CANDIDATE_THRESHOLDS:
#     leaf_correct = sum(
#         1 for p in positive_records
#         if p['confidence'] >= thresh and p['class_name'] in LEAF_TERMS
#     )
#     leaf_total = len(positive_records)

#     neg_false_positive = 0
#     neg_total = 0
#     for v in neg_results.values():
#         if 'detections' not in v:
#             continue
#         neg_total += 1
#         call = get_top_call_at_threshold(v['detections'], thresh)
#         if call in LEAF_TERMS:
#             neg_false_positive += 1

#     leaf_recall = leaf_correct / leaf_total * 100 if leaf_total else 0
#     fp_rate = neg_false_positive / neg_total * 100 if neg_total else 0

#     print(f"{thresh:<10} {leaf_recall:<15.1f} {fp_rate:<30.1f}")

In [26]:
# Cell 17 (final) — Save finalized Stage 0 config to Drive
STAGE0_CONFIG_PATH = CONFIG_DIR / 'stage0_config.json'

stage0_config = {
    'vocab': VOCAB,
    'leaf_terms': list(LEAF_TERMS),
    'conf_threshold': 0.08,
    'model_filename': MODEL_FILENAME,
    'box_selection_rule': 'leaf_priority',  # any leaf-term detection above threshold wins, regardless of rank
    'crop_padding_pct': 0.08,
    'min_box_area_ratio': 0.02,  # not yet independently validated — flagged earlier as a placeholder default
}

with open(STAGE0_CONFIG_PATH, 'w') as f:
    json.dump(stage0_config, f, indent=2)

print("Saved Stage 0 config:")
print(json.dumps(stage0_config, indent=2))

Saved Stage 0 config:
{
  "vocab": [
    "leaf",
    "plant leaf",
    "tree leaf",
    "grass leaf",
    "blade leaf",
    "crop leaf",
    "leaf in hand",
    "human hand",
    "person",
    "face",
    "sky",
    "soil",
    "ground",
    "mobile phone",
    "shoe",
    "clothing",
    "vehicle",
    "animal",
    "tool",
    "building",
    "wall",
    "sack of grain"
  ],
  "leaf_terms": [
    "plant leaf",
    "leaf",
    "crop leaf",
    "blade leaf",
    "tree leaf",
    "grass leaf",
    "leaf in hand"
  ],
  "conf_threshold": 0.08,
  "model_filename": "yoloe-11s-seg.pt",
  "box_selection_rule": "leaf_priority",
  "crop_padding_pct": 0.08,
  "min_box_area_ratio": 0.02
}


# Section E — the gate + crop function

In [27]:
# Cell 18 (final) — Stage 0 gate + crop function, using leaf-priority resolution
def stage0_leaf_gate(image_path, model, config):
    """
    Runs Stage 0 leaf detection on an image.

    Returns one of:
      {"status": "leaf", "crop": np.ndarray, "confidence": float, "box": [x1,y1,x2,y2]}
      {"status": "rejected", "detected_object": str, "confidence": float}
      {"status": "no_object_detected"}
    """
    conf_threshold = config['conf_threshold']
    leaf_terms = set(config['leaf_terms'])
    min_area_ratio = config['min_box_area_ratio']
    padding_pct = config['crop_padding_pct']

    img = np.array(Image.open(image_path).convert('RGB'))
    h, w = img.shape[:2]
    img_area = h * w

    results = model.predict(image_path, conf=0.01, verbose=False)  # low conf, filter via resolve_call
    r = results[0]

    if len(r.boxes) == 0:
        return {"status": "no_object_detected"}

    candidates = []
    for box in r.boxes:
        cls_id = int(box.cls.cpu().numpy()[0])
        conf = float(box.conf.cpu().numpy()[0])
        xyxy = box.xyxy.cpu().numpy()[0].tolist()
        x1, y1, x2, y2 = xyxy
        box_area = (x2 - x1) * (y2 - y1)
        area_ratio = box_area / img_area

        if area_ratio < min_area_ratio:
            continue

        candidates.append({
            'class_name': model.names[cls_id],
            'confidence': conf,
            'box': xyxy,
        })

    if not candidates:
        return {"status": "no_object_detected"}

    top = resolve_call(candidates, leaf_terms, conf_threshold)

    if top is None:
        return {"status": "no_object_detected"}

    if top['class_name'] not in leaf_terms:
        return {
            "status": "rejected",
            "detected_object": top['class_name'],
            "confidence": round(top['confidence'], 4),
        }

    x1, y1, x2, y2 = top['box']
    box_w, box_h = x2 - x1, y2 - y1
    pad_x, pad_y = box_w * padding_pct, box_h * padding_pct

    x1p = max(0, int(x1 - pad_x))
    y1p = max(0, int(y1 - pad_y))
    x2p = min(w, int(x2 + pad_x))
    y2p = min(h, int(y2 + pad_y))

    crop = img[y1p:y2p, x1p:x2p]

    return {
        "status": "leaf",
        "crop": crop,
        "confidence": round(top['confidence'], 4),
        "box": [x1p, y1p, x2p, y2p],
    }

In [28]:
# Cell 19 — Sanity test stage0_leaf_gate on sample images
with open(STAGE0_CONFIG_PATH, 'r') as f:
    stage0_config = json.load(f)

test_samples = {
    'leaf_example': all_ood_images[0] if all_ood_images else None,
    'negative_example': Path(neg_manifest[0]['path']) if neg_manifest else None,
}

for label, path in test_samples.items():
    if path is None:
        continue
    result = stage0_leaf_gate(str(path), model, stage0_config)
    print(f"\n{label} ({path.name}):")
    for k, v in result.items():
        if k == 'crop':
            print(f"  crop shape: {v.shape}")
        else:
            print(f"  {k}: {v}")


leaf_example (Apple_1.png):
  status: leaf
  crop shape: (513, 822, 3)
  confidence: 0.2203
  box: [0, 42, 822, 555]

negative_example (000a1249af2bc5f0.jpg):
  status: rejected
  detected_object: person
  confidence: 0.3357


In [29]:
# Cell 19b — Broader sanity check across a small mixed sample
import random

random.seed(42)
sample_leaf_paths = random.sample(all_ood_images, min(5, len(all_ood_images)))
sample_neg_paths = random.sample([Path(e['path']) for e in neg_manifest], min(5, len(neg_manifest)))

print("=== Leaf examples ===")
for path in sample_leaf_paths:
    result = stage0_leaf_gate(str(path), model, stage0_config)
    print(f"{path.name:30s} -> status={result['status']:15s} conf={result.get('confidence', 'N/A')}")

print("\n=== Negative examples ===")
for path in sample_neg_paths:
    result = stage0_leaf_gate(str(path), model, stage0_config)
    print(f"{path.name:30s} -> status={result['status']:15s} conf={result.get('confidence', 'N/A')}")

=== Leaf examples ===
Screenshot 2026-06-29 at 10.49.04 AM.png -> status=leaf            conf=0.4699
Screenshot 2026-06-16 at 6.28.46 PM.png -> status=leaf            conf=0.7537
IMG_4139.JPG                   -> status=rejected        conf=0.1207
IMG_4166.WEBP                  -> status=leaf            conf=0.4367
Screenshot 2026-06-29 at 11.50.26 AM.png -> status=leaf            conf=0.5273

=== Negative examples ===
01756a5b593e1de1.jpg           -> status=rejected        conf=0.8934
001997021f01f208.jpg           -> status=rejected        conf=0.8035
01c9c7321fe8e116.jpg           -> status=rejected        conf=0.1717
006f87bf928f9ba3.jpg           -> status=rejected        conf=0.8873
2443f4715e5f54d5.jpg           -> status=rejected        conf=0.2435
